## MULTI DATASET RECOMMENDATION

In [1]:
import numpy as np
import pandas as pd
from scipy.sparse import load_npz, issparse
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = [12, 5]

In [2]:
print('Loading vectors, sentiments, and topics...')
PATH_FEAT = "../../outputs/results/"
PATH_RAW  = "../../data/processed/multilingual_clean.csv"
X_tfidf = load_npz(f"{PATH_FEAT}multi_tfidf.npz")
X_sbert = np.load(f"{PATH_FEAT}multi_sbert.npy")
df = pd.read_csv(PATH_RAW)
df['pred_sentiment'] = df.get('pred_sentiment', df['sentiment'])
df['lda_topic'] = df.get('lda_topic', df.get('bert_topic', 0))
df['bert_topic'] = df.get('bert_topic', df.get('lda_topic', 0))
sentiment_to_score = {'positive': 1.0, 'neutral': 0.5, 'negative': 0.0}
df['sent_score'] = df['pred_sentiment'].map(sentiment_to_score).fillna(0.5)
print(f"Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")

Loading vectors, sentiments, and topics...
Rows: 4500
Columns: ['review_id', 'text', 'language', 'rating', 'sentiment', 'product_category', 'text_clean_classique', 'text_clean_light', 'pred_sentiment', 'lda_topic', 'bert_topic', 'sent_score']


In [3]:
def compute_recommendation_scores(query_idx, X, strategy='hybrid',
                                  w_sim=0.5, w_sent=0.3, w_topic=0.2,
                                  topic_col='bert_topic', boost_positive=True):
    query_vec = X[query_idx]
    if not issparse(query_vec):
        query_vec = np.atleast_2d(query_vec)
    sim_scores = cosine_similarity(query_vec, X).flatten()
    sim_scores[query_idx] = -1
    sim_norm = np.clip(sim_scores, 0, 1)
    if strategy in ['sentiment', 'hybrid']:
        q_sent = df.iloc[query_idx]['pred_sentiment']
        sent_match = (df['pred_sentiment'] == q_sent).astype(float).to_numpy()
        sent_boost = np.zeros(len(df))
        if boost_positive:
            sent_boost[df['pred_sentiment'] == 'positive'] = 0.2
            sent_boost[df['pred_sentiment'] == 'negative'] = -0.2
        sent_final = sent_match + sent_boost
    else:
        sent_final = np.zeros(len(df))
    if strategy in ['topic', 'hybrid']:
        q_topic = df.iloc[query_idx][topic_col]
        topic_match = (df[topic_col] == q_topic).astype(float).to_numpy()
    else:
        topic_match = np.zeros(len(df))
    final_scores = w_sim * sim_norm + w_sent * sent_final + w_topic * topic_match
    final_scores[query_idx] = -1
    top_indices = np.argsort(final_scores)[::-1]
    return top_indices, {
        'sim': sim_scores,
        'sent': sent_final,
        'topic': topic_match,
        'final': final_scores
    }

In [4]:
def run_baseline(X, name, query_idx, top_k=5):
    print(f"\nBASELINE: {name}")
    idx, scores = compute_recommendation_scores(query_idx, X, strategy='baseline')
    top_k_idx = idx[:top_k]
    for i, rec_idx in enumerate(top_k_idx, 1):
        print(f"{i}. [{scores['sim'][rec_idx]:.3f}] {df.iloc[rec_idx]['text_clean_light'][:70]}...")
    return top_k_idx
query_idx = 42
print(f"Query ({query_idx}): {df.iloc[query_idx]['text_clean_light'][:80]}...")
baseline_tfidf = run_baseline(X_tfidf, 'TF-IDF', query_idx)
baseline_sbert = run_baseline(X_sbert, 'SBERT Embeddings', query_idx)

Query (42): القنبلة ضيقة جدا على رأسها يجب أن تكون قياسية أنا خيبة أمل من الشراء...

BASELINE: TF-IDF
1. [1.000] القنبلة ضيقة جدا على رأسها يجب أن تكون قياسية أنا خيبة أمل من الشراء...
2. [1.000] القنبلة ضيقة جدا على رأسها يجب أن تكون قياسية أنا خيبة أمل من الشراء...
3. [1.000] القنبلة ضيقة جدا على رأسها يجب أن تكون قياسية أنا خيبة أمل من الشراء...
4. [0.239] جوميا أيضا لا ثقة ولا حماية المستهلك الخبرة السيئة مع شراء جوميا يجب أ...
5. [0.237] لا توجد بلوتوث، لقد كنت في رحلة بعد استقبال التلفاز، خيبة أمل كبيرة....

BASELINE: SBERT Embeddings
1. [1.000] القنبلة ضيقة جدا على رأسها يجب أن تكون قياسية أنا خيبة أمل من الشراء...
2. [1.000] القنبلة ضيقة جدا على رأسها يجب أن تكون قياسية أنا خيبة أمل من الشراء...
3. [1.000] القنبلة ضيقة جدا على رأسها يجب أن تكون قياسية أنا خيبة أمل من الشراء...
4. [0.643] le bonnet est trop serré sur tête il faut qu'il soit standard je suis ...
5. [0.643] le bonnet est trop serré sur tête il faut qu'il soit standard je suis ...


In [5]:
def run_enhanced(X, strategy, query_idx, top_k=5, **kwargs):
    print(f"\nSTRATEGY: {strategy.upper()}")
    idx, scores = compute_recommendation_scores(query_idx, X, strategy=strategy, **kwargs)
    top_k_idx = idx[:top_k]
    for i, rec_idx in enumerate(top_k_idx, 1):
        sim = scores['sim'][rec_idx]
        sent = scores['sent'][rec_idx]
        topic = scores['topic'][rec_idx]
        print(f"{i}. [Sim:{sim:.2f} | Sent:{sent:.2f} | Topic:{topic:.1f}] {df.iloc[rec_idx]['text_clean_light'][:60]}...")
    return top_k_idx
sent_rec = run_enhanced(X_sbert, 'sentiment', query_idx, w_sim=0.6, w_sent=0.4)
topic_rec = run_enhanced(X_sbert, 'topic', query_idx, w_sim=0.6, w_topic=0.4)
hybrid_rec = run_enhanced(X_sbert, 'hybrid', query_idx, w_sim=0.5, w_sent=0.3, w_topic=0.2)


STRATEGY: SENTIMENT
1. [Sim:1.00 | Sent:0.80 | Topic:0.0] القنبلة ضيقة جدا على رأسها يجب أن تكون قياسية أنا خيبة أمل م...
2. [Sim:1.00 | Sent:0.80 | Topic:0.0] القنبلة ضيقة جدا على رأسها يجب أن تكون قياسية أنا خيبة أمل م...
3. [Sim:1.00 | Sent:0.80 | Topic:0.0] القنبلة ضيقة جدا على رأسها يجب أن تكون قياسية أنا خيبة أمل م...
4. [Sim:0.64 | Sent:0.80 | Topic:0.0] le bonnet est trop serré sur tête il faut qu'il soit standar...
5. [Sim:0.64 | Sent:0.80 | Topic:0.0] le bonnet est trop serré sur tête il faut qu'il soit standar...

STRATEGY: TOPIC
1. [Sim:1.00 | Sent:0.00 | Topic:1.0] القنبلة ضيقة جدا على رأسها يجب أن تكون قياسية أنا خيبة أمل م...
2. [Sim:1.00 | Sent:0.00 | Topic:1.0] القنبلة ضيقة جدا على رأسها يجب أن تكون قياسية أنا خيبة أمل م...
3. [Sim:1.00 | Sent:0.00 | Topic:1.0] القنبلة ضيقة جدا على رأسها يجب أن تكون قياسية أنا خيبة أمل م...
4. [Sim:0.64 | Sent:0.00 | Topic:1.0] le bonnet est trop serré sur tête il faut qu'il soit standar...
5. [Sim:0.64 | Sent:0.00 | Topic:1.0] le bon

In [6]:
def explain_recommendation(query_idx, rec_idx, scores):
    q_sent = df.iloc[query_idx]['pred_sentiment']
    r_sent = df.iloc[rec_idx]['pred_sentiment']
    q_topic = df.iloc[query_idx]['bert_topic']
    r_topic = df.iloc[rec_idx]['bert_topic']
    reasons = []
    if scores['sim'][rec_idx] > 0.6: reasons.append('strong text similarity')
    if scores['sent'][rec_idx] > 0: reasons.append('sentiment aligned/positive')
    if scores['topic'][rec_idx] == 1: reasons.append('same topic')
    if r_sent == 'positive': reasons.append('positive review (boosted)')
    elif r_sent == 'negative': reasons.append('negative review (penalized)')
    print(f"Recommended because: {', '.join(reasons)}")
print('Explain top 1 hybrid:')
explain_recommendation(query_idx, hybrid_rec[0], compute_recommendation_scores(query_idx, X_sbert, 'hybrid')[1])

Explain top 1 hybrid:
Recommended because: strong text similarity, sentiment aligned/positive, same topic, negative review (penalized)


In [7]:
print('Category recommendation')
cat_scores = df.groupby('product_category').agg({
    'pred_sentiment': lambda x: (x == 'positive').mean(),
    'bert_topic': lambda x: x.mode()[0] if not x.mode().empty else None
}).rename(columns={'pred_sentiment': 'positive_ratio', 'bert_topic': 'dominant_topic'})
query_topic = df.iloc[query_idx]['bert_topic']
cat_scores['rec_score'] = 0.0
cat_scores.loc[cat_scores['dominant_topic'] == query_topic, 'rec_score'] += 0.6
cat_scores['rec_score'] += cat_scores['positive_ratio'] * 0.4
top_cats = cat_scores.sort_values('rec_score', ascending=False).head(3)
print(top_cats[['positive_ratio', 'dominant_topic', 'rec_score']].to_markdown())

Category recommendation
| product_category         |   positive_ratio |   dominant_topic |   rec_score |
|:-------------------------|-----------------:|-----------------:|------------:|
| Vêtements & Chaussures   |         1        |                0 |    1        |
| Maison, cuisine & bureau |         0.443782 |                0 |    0.777513 |
| baby                     |         0.333333 |                0 |    0.733333 |
